# Non-Linear Classification with NeuraCommand

This notebook demonstrates how to classify non-linearly separable datasets using neural networks with NeuraCommand.

## Overview

We use the `make_moons` dataset from scikit-learn, which generates two interleaving half circles that cannot be separated by a linear decision boundary. A multi-layer neural network with non-linear activation functions can learn the complex decision boundary needed to classify such data.

**Key Concepts:**
- Non-linear activation functions (ReLU, Sigmoid)
- Hidden layers for learning complex patterns
- Decision boundary visualization

In [ ]:
# Install if needed (uncomment)
# !pip install -e ..
# !pip install scikit-learn matplotlib

In [ ]:
import numpy as np
import sys
sys.path.insert(0, '../src')

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

from neura_command import Sequential, Dense, ReLU, Sigmoid, Adam, MSELoss, __version__

print(f"NeuraCommand v{__version__}")

## 1. Generate the Moons Dataset

The moons dataset creates two interleaving half circles - a classic example of non-linearly separable data.

In [ ]:
# Generate the moons dataset
n_samples = 500
noise = 0.2
random_state = 42

X, y = make_moons(n_samples=n_samples, noise=noise, random_state=random_state)

# Reshape y for neural network (needs to be 2D)
y = y.reshape(-1, 1).astype(np.float32)
X = X.astype(np.float32)

print(f"Dataset shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class distribution: {np.sum(y == 0)} class 0, {np.sum(y == 1)} class 1")

In [ ]:
# Visualize the raw dataset
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap=plt.cm.RdBu, edgecolors='black', s=50)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Moons Dataset - Non-Linearly Separable Data')
plt.legend(*scatter.legend_elements(), title='Classes')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice how no straight line can separate the two classes!")

## 2. Prepare the Data

Split into training and test sets, then standardize features for better training.

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state, stratify=y
)

# Standardize features for better training
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features shape: {X_train.shape}")

## 3. Build the Neural Network

We create a multi-layer network with:
- **Input**: 2 features (x, y coordinates)
- **Hidden layer 1**: 16 neurons with ReLU activation
- **Hidden layer 2**: 8 neurons with ReLU activation  
- **Output**: 1 neuron with Sigmoid activation (binary classification)

In [ ]:
# Build the model
model = Sequential(
    layers=[
        Dense(2, 16),      # 2 inputs -> 16 hidden units
        ReLU(),
        Dense(16, 8),      # 16 -> 8 hidden units
        ReLU(),
        Dense(8, 1),       # 8 -> 1 output
        Sigmoid(),         # Output between 0 and 1
    ],
    optimizer=Adam(learning_rate=0.01),
    loss=MSELoss(),
)

print(model.summary())

## 4. Train the Model

In [ ]:
# Train the model
history = model.train(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=True,
)

## 5. Visualize Training Progress

In [ ]:
# Plot training history
plt.figure(figsize=(10, 5))

plt.plot(history['loss'], label='Training Loss', color='blue')
if 'val_loss' in history:
    plt.plot(history['val_loss'], label='Validation Loss', color='orange')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Evaluate the Model

In [ ]:
# Make predictions
predictions_prob = model.predict(X_test)
predictions = (predictions_prob > 0.5).astype(int)

# Calculate accuracy
accuracy = accuracy_score(y_test, predictions)

print(f"Test Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=['Class 0', 'Class 1']))

## 7. Visualize the Decision Boundary

Let's see how the neural network learned to separate the two classes with a curved decision boundary.

In [ ]:
# Combine data for visualization
X_combined = np.vstack([X_train, X_test])
y_combined = np.vstack([y_train, y_test])

# Create a mesh grid
h = 0.02  # Step size
x_min, x_max = X_combined[:, 0].min() - 0.5, X_combined[:, 0].max() + 0.5
y_min, y_max = X_combined[:, 1].min() - 0.5, X_combined[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Get predictions for the mesh grid
grid_points = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
Z = model.predict(grid_points)
Z = Z.reshape(xx.shape)

# Plot
plt.figure(figsize=(12, 8))

# Plot decision boundary
plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=0.8, levels=50)
plt.colorbar(label='Prediction Probability')

# Plot decision boundary line
plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

# Plot data points
scatter = plt.scatter(
    X_combined[:, 0], X_combined[:, 1], c=y_combined.ravel(), 
    cmap=plt.cm.RdBu, edgecolors='black', s=50
)

plt.xlabel('Feature 1 (Standardized)')
plt.ylabel('Feature 2 (Standardized)')
plt.title(f'Non-Linear Decision Boundary (Accuracy: {accuracy * 100:.1f}%)')
plt.legend(*scatter.legend_elements(), title='Classes')
plt.tight_layout()
plt.show()

print("The black line shows the decision boundary learned by the neural network.")
print("Notice how it curves to separate the two interleaving half-moons!")

## 8. Experiment: Compare with Different Architectures

Let's see how different network architectures perform on this problem.

In [ ]:
architectures = {
    'Shallow (2->4->1)': [Dense(2, 4), ReLU(), Dense(4, 1), Sigmoid()],
    'Medium (2->16->8->1)': [Dense(2, 16), ReLU(), Dense(16, 8), ReLU(), Dense(8, 1), Sigmoid()],
    'Deep (2->32->16->8->1)': [Dense(2, 32), ReLU(), Dense(32, 16), ReLU(), Dense(16, 8), ReLU(), Dense(8, 1), Sigmoid()],
}

results = {}

for name, layers in architectures.items():
    model = Sequential(
        layers=layers,
        optimizer=Adam(learning_rate=0.01),
        loss=MSELoss(),
    )
    model.train(X_train, y_train, epochs=100, batch_size=32, verbose=False)
    pred = (model.predict(X_test) > 0.5).astype(int)
    acc = accuracy_score(y_test, pred) * 100
    results[name] = acc
    print(f"{name:30s}: {acc:.1f}% accuracy")

print("\nAll architectures can learn non-linear decision boundaries!")

## Summary

In this notebook, we demonstrated:

1. **Non-linear classification** - Using the `make_moons` dataset which cannot be separated by a straight line
2. **Data preprocessing** - Standardization for better neural network training
3. **Model architecture** - Hidden layers with ReLU activations enable learning curved decision boundaries
4. **Training and evaluation** - Full end-to-end workflow with NeuraCommand
5. **Visualization** - Decision boundary shows how the network separates classes

**Key Takeaways:**
- Neural networks with hidden layers can learn complex, non-linear patterns
- ReLU activation introduces non-linearity needed for curved boundaries
- Sigmoid output is ideal for binary classification (outputs probability 0-1)
- Feature standardization helps training converge faster

**Next Steps:**
- Try the `make_circles` dataset for another non-linear problem
- Experiment with different optimizers (SGD, RMSprop)
- Add dropout for regularization
- Extend to multi-class classification with Softmax